# Notebook 02 — Why MCP servers

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

If MCP servers are just Lambdas, why do I need them? Why can't agents call AWS services directly?

This notebook answers that question with working code. By the end, you will have
seen the contract that MCP servers enforce — and you will understand why that
contract, not a more direct connection, is the right choice for a system that
needs to evolve without breaking the agents that depend on it.

## Key terms for this notebook

| Term | What it is |
|------|------------|
| **MCP server** | A small service that exposes a fixed, named set of operations — a stable typed interface — over a capability. The service handles the implementation details (which AWS service, which endpoint, which IAM role); callers only see the operation names and their input/output shapes. MCP stands for Model Context Protocol. |
| **Operation schema** | The declared input and output types of a single MCP operation. For example, `query(sparql: string, persona_claim: string) → {rows: array, execution_time_ms: number}`. The schema is the contract. The implementation behind it is free to change. |
| **Capability surface** | The full set of operations that agents in a system can call. In ATLAS, five MCP servers define the capability surface: SPARQL, SHACL validation, Entity Resolution, FIBO introspection, and registry discovery. |
| **Persona claim** | A string that identifies who is asking — for example, `atlas-consumer-banker`. MCP servers use the persona claim to scope what data the operation is allowed to return. An agent passes its caller's persona claim through to the MCP server; the server enforces the boundary. |
| **Stable typed interface** | An interface whose operation names and schemas do not change even when the implementation behind them does. Stability is what makes agents forward-compatible: they were written against the schema, not against the implementation, so they keep working when the implementation changes. |

## A stable surface for an unstable world

Notebook 01 built `ask_graph()`: a function that translates a natural-language
question into a SPARQL query and runs it against the Neptune knowledge graph. That
function works, and it works correctly — but it carries a hidden cost. Inside
`ask_graph()`, there is a `NeptuneClient` constructed with a specific endpoint, a
specific port, and specific error-handling assumptions. If the Neptune cluster
moves — if a new endpoint is deployed, if the connection pattern switches from
direct HTTPS to Ontop (the SPARQL-over-relational translation layer on ECS), if
the graph tier changes from the SLGD (Semantic Layer Graph Database) to the LGD
(Lexical Graph Database) for a diagnostic query — every agent that contains that
connection logic must be updated. That is the first problem an MCP server solves:
it separates the *what* (run a SPARQL query) from the *how* (which cluster, which
tier, which connection pattern).

The second problem is persona enforcement. The `atlas-sparql-mcp` server accepts
a `persona_claim` parameter on every operation. That claim is translated, inside
the server, into a Lake Formation (AWS Lake Formation) scope that filters which
rows and columns the query is allowed to return. A Consumer Banker's claim
produces a scope that covers their assigned book of clients. A BSA Analyst's claim
produces a scope that includes compliance-restricted fields the Consumer Banker
cannot see. If every agent enforced this scope independently — each one knowing
the Lake Formation tag taxonomy, the mapping from persona to tag, and the Neptune
connection path — the enforcement would be inconsistent. A missing clause in one
agent would expose data it should not expose. The MCP server is the single place
where that enforcement lives, so it can be tested once and trusted everywhere.

This is why ATLAS has five MCP servers rather than five agents that each call AWS
services directly. Each server wraps one capability: SPARQL over Neptune, SHACL
(Shapes Constraint Language) validation, Entity Resolution identity lookup, FIBO
(Financial Industry Business Ontology) class introspection, and registry discovery.
Together they form the capability surface that every Phase 1 agent operates against.
The agents are thin: they receive a request, identify which MCP operations to call,
call them in sequence, and return the assembled result. The agents do not know which
Neptune cluster is running, which Lake Formation tags are in effect, or which version
of the FIBO ontology is loaded. The MCP servers know those things. The agents know
only the operation schemas.

This is also the foundation of Thesis 1 from the ATLAS architecture: registry-first
agent discovery. For a UI to show a capability palette — a list of actions the
current user is allowed to invoke — it needs to know which capabilities exist. The
Agent Registry answers that question by returning a persona-scoped list of registered
MCP operations. But the registry can only answer that question because each MCP
server has a descriptor: a JSON file that declares its operations, their schemas,
and which personas are allowed to discover them. The MCP server descriptor is what
makes the capability surface enumerable. Without it, capabilities are implicit —
encoded in agent source code, invisible to the registry, and invisible to the UI.
With it, every capability is named, typed, and governable.

The production MCP servers in ATLAS are **AgentCore Runtimes**, not plain Lambda
functions. An AgentCore Runtime runs the same thin Python handler — but it runs
inside the AWS Bedrock AgentCore service, which assigns each server a stable
ARN address and exposes invocation metrics through CloudWatch. The CDK stack in
`use-case-applications/cdk/` deploys all five MCP servers as AgentCore Runtimes
and wires them with the correct IAM policies, environment variables, and Neptune
endpoints.

Deploying on AgentCore instead of raw Lambda makes the invocation surface uniform
across all five servers. Every caller — every agent, every AppSync proxy — uses
the same SDK call regardless of which server it is invoking:

```python
response = boto3.client("bedrock-agentcore").invoke_agent_runtime(
    agentRuntimeArn=SPARQL_MCP_ARN,
    payload=json.dumps({"operation": "query", ...}).encode(),
    contentType="application/json",
)
result = json.loads(response["response"].read())
```

The ARN is the only thing that changes per server. The method name, the payload
encoding, the response key — identical across all five. An agent that calls five
different MCP servers for five different capabilities uses the same invocation
pattern five times. That uniformity is not cosmetic: it means a new MCP server can
be integrated by any agent that already calls an existing one, with no new SDK
pattern to learn. It also means that a monitoring alert on `invoke_agent_runtime`
errors covers the entire MCP layer simultaneously, with no per-server alerting
configuration needed.

This notebook does not deploy anything. What it does is demonstrate the contract
that each Runtime honours: the operation schema. You will implement
`sparql_mcp_query()` as a local Python function, run it against the real SLGD,
and verify that its response shape is exactly what the descriptor declares. When
the Runtime is deployed later, it will honour the same schema — and the agents
written against it in notebook 03 will keep working without modification.

In [ ]:
import sys
import os
import json
import time

# Workshop 1's shared helpers — same path as notebook 01.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

import boto3
from pathlib import Path

from atlas_neptune import NeptuneClient
from atlas_sparql import build_prefixes, validate, AtlasSPARQLError

STACK_NAME = "atlas-neptune-twotier"
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

try:
    REPO_ROOT = Path(__file__).resolve().parents[3]
except NameError:
    REPO_ROOT = Path("../../..").resolve()

# Retrieve Neptune endpoint from CloudFormation — same pattern as notebook 00 and 01.
cfn = boto3.client("cloudformation", region_name=AWS_REGION)
response = cfn.describe_stacks(StackName=STACK_NAME)
outputs = {o["OutputKey"]: o["OutputValue"] for o in response["Stacks"][0].get("Outputs", [])}

slgd = NeptuneClient(
    endpoint=outputs["SLGDEndpoint"],
    port=int(outputs.get("SLGDPort", 8182)),
)

print(f"SLGD endpoint: {outputs['SLGDEndpoint']}")
print("Setup complete.")

In [ ]:
# Build cell 1 — A direct query against the SLGD using NeptuneClient.
#
# This is how an agent would call the graph *without* an MCP server.
# Notice what the agent has to know to make this work:
#   - The NeptuneClient class and where to import it from
#   - The SLGD endpoint (from CloudFormation outputs, env vars, or hardcoded)
#   - The port number
#   - The SPARQL prefix block
#   - How to parse the result rows
#   - How to handle connection errors
#
# Every agent that queries the graph must carry all of this. When the cluster
# moves, or the connection pattern changes, every agent must be updated.
# That is the problem the next cell removes.

COVERAGE_GAP_QUERY = (
    build_prefixes()
    + "\n"
    + """
SELECT ?customer ?name WHERE {
    ?customer a atlas:Customer .
    OPTIONAL { ?customer atlas:fullName ?name . }
    FILTER NOT EXISTS {
        ?customer atlas:hasAdvisor ?rel .
    }
}
LIMIT 5
"""
)

rows = slgd.query(COVERAGE_GAP_QUERY)

print("Direct NeptuneClient query — customers with no wealth advisor assigned:")
print()
if rows:
    for row in rows:
        print(f"  {row}")
else:
    print("  (no rows — check that Workshop 1 module 4 ran successfully)")
print()
print("The query ran. But the agent that ran it had to know the endpoint,")
print("the port, the NeptuneClient import path, and the prefix block.")
print("The next cell wraps all of that behind a single function call.")

The query answered the question correctly. But it required the caller to manage
Neptune connection details, prefix declarations, and row parsing directly. In a
system with five agents — each of which queries the graph — that coupling appears
five times. The next cell wraps all of it behind the `sparql_mcp_query()` function,
whose signature matches the `query` operation declared in the `atlas-sparql-mcp`
descriptor. That is the MCP contract: a stable interface the caller can depend on
regardless of what changes behind it.

In [ ]:
# Build cell 2 — sparql_mcp_query(): the MCP-style contract.
#
# This function mirrors the `query` operation from atlas-sparql-mcp.json:
#
#   query(
#     sparql: string,
#     persona_claim: string,
#     graph_tier: enum[slgd, lgd]
#   ) → { rows: array, execution_time_ms: number }
#
# Two things this function does that the direct NeptuneClient call did not:
#
#   1. Validate the SPARQL before sending it. atlas_sparql.validate() catches
#      syntax errors and boundary violations (e.g. a query that would write
#      probabilistic-opaque data without an explainability attribute). This
#      is the enforcement point for ATLAS's deterministic boundary.
#
#   2. Reject anonymous calls. Every invocation must carry a persona_claim.
#      In production, the Lambda passes this claim to Lake Formation for scope
#      translation. Here we assert it is present so the contract is honoured
#      even in the local demonstration.
#
# What this function does *not* do: reason about the query content, modify the
# SPARQL, or decide which results to return based on the persona. It enforces
# the boundary and delegates everything else to the graph.

def sparql_mcp_query(
    sparql: str,
    persona_claim: str,
    graph_tier: str = "slgd",
) -> dict:
    """Local demonstration of the atlas-sparql-mcp `query` operation.

    Returns
    -------
    dict with keys:
        rows              — list of result row dicts
        execution_time_ms — wall-clock time for the Neptune round-trip

    On error returns:
        error             — human-readable description
        error_type        — one of: validation_error, persona_error, query_error
    """
    # The MCP boundary: no anonymous queries.
    if not persona_claim or not persona_claim.strip():
        return {
            "error": "persona_claim is required. Pass the caller's persona claim.",
            "error_type": "persona_error",
        }

    # Validate SPARQL syntax and boundary rules before touching the network.
    try:
        validate(sparql)
    except AtlasSPARQLError as exc:
        return {
            "error": str(exc),
            "error_type": "validation_error",
        }

    # In production the Lambda would translate persona_claim → Lake Formation
    # scope and choose the graph tier based on graph_tier. Here we use the
    # SLGD NeptuneClient that was configured in the setup cell.
    try:
        start = time.monotonic()
        rows = slgd.query(sparql)
        elapsed_ms = round((time.monotonic() - start) * 1000)
    except Exception as exc:
        return {
            "error": f"Neptune query failed: {exc}",
            "error_type": "query_error",
        }

    return {
        "rows": rows,
        "execution_time_ms": elapsed_ms,
    }


print("sparql_mcp_query() defined.")
print()
print("Operation schema (from atlas-sparql-mcp.json):")
print("  Input:  sparql: string, persona_claim: string, graph_tier: enum[slgd,lgd]")
print("  Output: { rows: array, execution_time_ms: number }")
print("  Error:  { error: string, error_type: string }")

In [ ]:
# Build cell 3 — Call sparql_mcp_query() with three different inputs.
#
# Three structurally different queries: a coverage gap lookup, a household
# count, and a simple class enumeration. The queries differ; the response
# shape — rows + execution_time_ms — is identical across all three.
# That shape consistency is the contract. Callers can depend on it.

CALLS = [
    {
        "label": "Coverage gap — customers with no advisor",
        "sparql": build_prefixes() + """
SELECT ?customer WHERE {
    ?customer a atlas:Customer .
    FILTER NOT EXISTS { ?customer atlas:hasAdvisor ?rel . }
}
LIMIT 3
""",
        "persona_claim": "atlas-consumer-banker",
    },
    {
        "label": "Household count",
        "sparql": build_prefixes() + """
SELECT (COUNT(?hh) AS ?total) WHERE {
    ?hh a atlas:Household .
}
""",
        "persona_claim": "atlas-consumer-banker",
    },
    {
        "label": "Advisory relationship sample",
        "sparql": build_prefixes() + """
SELECT ?rel ?customer ?advisor WHERE {
    ?rel a atlas:AdvisoryRelationship ;
         atlas:servicesCustomer ?customer ;
         atlas:servedByAdvisor  ?advisor .
}
LIMIT 3
""",
        "persona_claim": "atlas-wealth-advisor",
    },
]

for call in CALLS:
    result = sparql_mcp_query(call["sparql"], call["persona_claim"])
    print(f"--- {call['label']} ---")
    print(f"persona_claim:     {call['persona_claim']}")
    print(f"response keys:     {sorted(result.keys())}")
    print(f"execution_time_ms: {result.get('execution_time_ms')}")
    print(f"rows returned:     {len(result.get('rows', []))}")
    if result.get('rows'):
        print(f"first row:         {result['rows'][0]}")
    print()

## Verification

Two properties of `sparql_mcp_query()` must hold before this contract is useful to
any agent that depends on it. First, the response shape must be consistent — the
same keys must appear regardless of what the query returns. Second, a malformed
query must produce a structured error response, not an unhandled exception. An agent
that cannot distinguish between *"query returned zero rows"* and *"the function
raised an exception"* cannot handle errors safely. Both cells below print their
findings before asserting, so you can inspect what the function actually returned
before deciding whether the assertion is correct.

In [ ]:
# Verification cell 1 — Response shape is consistent across 5 invocations.
#
# We run the same query 5 times and check that:
#   (a) every response has exactly the keys `rows` and `execution_time_ms`
#   (b) `rows` is a list in every response
#   (c) `execution_time_ms` is a number in every response
#
# We do not assert that execution_time_ms is identical across runs — timing
# varies legitimately. We assert that the *shape* is identical.

RUNS = 5
SHAPE_QUERY = build_prefixes() + """
SELECT ?customer WHERE {
    ?customer a atlas:Customer .
    FILTER NOT EXISTS { ?customer atlas:hasAdvisor ?rel . }
}
LIMIT 1
"""
PERSONA = "atlas-consumer-banker"
EXPECTED_KEYS = {"rows", "execution_time_ms"}

print(f"Running sparql_mcp_query() {RUNS} times and inspecting response shape...")
print()

results = [sparql_mcp_query(SHAPE_QUERY, PERSONA) for _ in range(RUNS)]

shape_failures = []
for i, result in enumerate(results):
    actual_keys = set(result.keys())
    rows_is_list = isinstance(result.get("rows"), list)
    time_is_number = isinstance(result.get("execution_time_ms"), (int, float))
    ok = (actual_keys == EXPECTED_KEYS) and rows_is_list and time_is_number
    status = "[PASS]" if ok else "[FAIL]"
    print(f"  Run {i+1}: {status}  keys={sorted(actual_keys)}  "
          f"rows={type(result.get('rows')).__name__}  "
          f"execution_time_ms={result.get('execution_time_ms')}")
    if not ok:
        shape_failures.append(i + 1)

print()

if shape_failures:
    print(f"VERIFICATION FAILED: Runs {shape_failures} returned unexpected response shape.")
    print("Expected keys: rows (list), execution_time_ms (number).")
    print("If 'error' appears instead, sparql_mcp_query() hit an error path. Check")
    print("that the SLGD endpoint is reachable and that the query is valid SPARQL.")
    print("If extra keys appear, the function definition in cell 06 may have been modified.")

assert not shape_failures, (
    f"Response shape inconsistent on runs {shape_failures}. "
    "Expected keys: rows (list), execution_time_ms (number)."
)

print(f"[PASS] All {RUNS} responses have the correct shape.")
print("The operation schema holds. Agents can depend on this interface.")

In [ ]:
# Verification cell 2 — Malformed SPARQL returns a structured error, not an exception.
#
# An agent that calls sparql_mcp_query() must be able to handle the case where
# the SPARQL is invalid. If the function raises an exception, the agent's call
# stack unwinds and the error is invisible to any caller further up the chain.
# If the function returns a structured dict with an `error` key, the agent can
# inspect it, log it, and return a graceful response to the user.
#
# We pass three forms of bad input and check that each one returns a structured
# error dict rather than raising.

BAD_INPUTS = [
    {
        "label": "Syntactically invalid SPARQL",
        "sparql": "THIS IS NOT SPARQL",
        "persona_claim": "atlas-consumer-banker",
        "expected_error_type": "validation_error",
    },
    {
        "label": "Missing persona claim",
        "sparql": build_prefixes() + "SELECT ?x WHERE { ?x a atlas:Customer . } LIMIT 1",
        "persona_claim": "",
        "expected_error_type": "persona_error",
    },
    {
        "label": "Boundary violation — probabilistic-opaque INSERT without explainability",
        "sparql": (
            build_prefixes()
            + "INSERT DATA { atlas:testNode atlas:probabilisticOpaque true . }"
        ),
        "persona_claim": "atlas-ontology-steward",
        "expected_error_type": "validation_error",
    },
]

print("Testing sparql_mcp_query() with malformed inputs...")
print()

error_failures = []

for case in BAD_INPUTS:
    raised = False
    result = None
    try:
        result = sparql_mcp_query(case["sparql"], case["persona_claim"])
    except Exception as exc:
        raised = True
        result = {"raised": str(exc)}

    is_structured = (
        not raised
        and isinstance(result, dict)
        and "error" in result
        and result.get("error_type") == case["expected_error_type"]
    )
    status = "[PASS]" if is_structured else "[FAIL]"

    print(f"  {status} {case['label']}")
    print(f"         result: {result}")
    if not is_structured:
        error_failures.append(case["label"])

print()

if error_failures:
    print("VERIFICATION FAILED: The following cases did not return a structured error dict:")
    for label in error_failures:
        print(f"  - {label}")
    print("Expected: a dict with 'error' (string) and 'error_type' (string) keys.")
    print("Actual: either an exception was raised, or the dict was missing expected keys.")
    print("Check the error-handling branches in sparql_mcp_query() (cell 06).")
    print("Each except block should return a dict, not re-raise the exception.")

assert not error_failures, (
    f"Structured error contract failed for: {error_failures}. "
    "sparql_mcp_query() must return a dict with 'error' and 'error_type' keys "
    "rather than raising an exception."
)

print(f"[PASS] All {len(BAD_INPUTS)} malformed inputs returned structured error responses.")
print("Agents calling sparql_mcp_query() can safely inspect the return value")
print("rather than wrapping every call in a try/except.")

## What just changed

You have a local implementation of the `atlas-sparql-mcp` operation schema: a
function that accepts a SPARQL query and a persona claim, validates both, runs the
query against the SLGD, and returns a response whose shape is stable regardless of
query content. That stability is the contract — the thing agents will be written
against in notebook 03.

The production version of this contract is an AgentCore Runtime deployed by the
CDK stack in `use-case-applications/cdk/`. It runs the same Python handler against
the real Neptune SLGD, with SigV4 signing for the underlying AWS calls and persona
claims translated to Lake Formation scopes for row-level access control. Four more
Runtimes follow the same pattern for SHACL validation, Entity Resolution, FIBO
introspection, and registry discovery. All five are invoked identically —
`bedrock-agentcore.invoke_agent_runtime()` with the Runtime's ARN and a JSON
payload — so adding a sixth server in a future phase requires no new invocation
pattern anywhere in the fleet.

What becomes possible next: once the MCP servers are registered, the registry knows
what capabilities exist and who is allowed to discover them. The UI does not need to
hardcode that list — it asks the registry, and the registry answers based on the
caller's persona. That is Thesis 1 of the ATLAS architecture, and it is what
notebook 03 builds.